In [1]:
import jax
import jax.numpy as jnp
import netket as nk
import netket.experimental as nkx
import numpy as np
from pyscf import gto, scf, fci
from flax import nnx
import optax
from functools import partial
from tqdm import tqdm
import matplotlib.pyplot as plt

print(f"JAX version: {jax.__version__}")
print(f"NetKet version: {nk.__version__}")

/opt/miniconda3/envs/Neural/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


JAX version: 0.5.3
NetKet version: 3.18


In [2]:
# H₂ 分子定义
bond_length = 1.4
geometry = [('H', (0., 0., 0.)), ('H', (bond_length, 0., 0.))]
mol = gto.M(atom=geometry, basis='STO-3G', verbose=0)
mf = scf.RHF(mol).run(verbose=0)

# FCI 精确基准
cisolver = fci.FCI(mf)
cisolver.nroots = 4
E_fcis, fcivec = cisolver.kernel()

print("="*60)
print("H₂ FCI 基准能量")
print("="*60)
for i, e in enumerate(E_fcis):
    exc = (e - E_fcis[0]) * 27.2114
    print(f"E{i} = {e:.8f} Ha  |  激发能: {exc:.4f} eV")

# NetKet 哈密顿量
ha = nkx.operator.from_pyscf_molecule(mol)

# Hilbert 空间
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=2,
    s=1/2,
    n_fermions_per_spin=(1,1),
)

print(f"\n希尔伯特空间大小: {hi.n_states}")
print(f"希尔伯特空间维度: {hi.size}")

H₂ FCI 基准能量
E0 = -1.01546825 Ha  |  激发能: 0.0000 eV
E1 = -0.87542794 Ha  |  激发能: 3.8107 eV
E2 = -0.42938376 Ha  |  激发能: 15.9482 eV
E3 = -0.26922131 Ha  |  激发能: 20.3064 eV

希尔伯特空间大小: 4
希尔伯特空间维度: 4


In [3]:
class SingleStateAnsatz(nnx.Module):
    def __init__(self, n_spin_orbitals: int, hidden_dim=16, *, rngs: nnx.Rngs):
        super().__init__()
        self.linear1 = nnx.Linear(n_spin_orbitals, hidden_dim, rngs=rngs, param_dtype=complex)
        self.linear2 = nnx.Linear(hidden_dim, hidden_dim, rngs=rngs, param_dtype=complex)
        self.output = nnx.Linear(hidden_dim, 1, rngs=rngs, param_dtype=complex)

    def __call__(self, x):
        h = nnx.tanh(self.linear1(x.astype(complex)))
        h = nnx.tanh(self.linear2(h))
        out = self.output(h)
        return jnp.squeeze(out)

def forward(GraphDef_State, x):
    """NetKet 采样器需要的 forward 函数"""
    log_psi, _ = nnx.call(GraphDef_State)(x)
    return log_psi

In [5]:
# NetKet 原生实现
rngs = nnx.Rngs(21)
model_netket = SingleStateAnsatz(n_spin_orbitals=4, hidden_dim=12, rngs=rngs)

# 设置采样器
edges = [(0, 1), (2, 3)]
g = nk.graph.Graph(edges=edges)
single_rule = nk.sampler.rules.FermionHopRule(hilbert=hi, graph=g)
sampler = nk.sampler.MetropolisSampler(hi, rule=single_rule, n_chains=16, sweep_size=32)

# 创建 MCState
vstate = nk.vqs.MCState(
    sampler=sampler,
    model=model_netket,
    n_samples=1000,
    n_discard_per_chain=10
)

# 设置 SR 和优化器
preconditioner = nk.optimizer.SR(diag_shift=1e-3, holomorphic=True)
optimizer = nk.optimizer.Sgd(0.1)

# 创建 VMC 驱动
vmc = nk.driver.VMC(ha, optimizer, variational_state=vstate, preconditioner=preconditioner)

print(f"\n{'='*70}")
print("NetKet 原生实现训练")
print(f"{'='*70}\n")

# 使用 NetKet 的 run 方法
vmc.run(400, out=None)  # out=None 表示不保存到文件

# 获取最终能量
energy_final_netket = vstate.expect(ha)

print(f"\n{'='*70}")
print(f"NetKet 训练完成！最终能量: {energy_final_netket.mean.real:.8f} Ha")
print(f"{'='*70}\n")

/opt/miniconda3/envs/Neural/lib/python3.11/site-packages/netket/vqs/mc/mc_state/state.py:300: UserWarning: n_samples=1000 (1000 per device/MPI rank) does not divide n_chains=16, increased to 1008 (1008 per device/MPI rank)
  self.n_samples = n_samples



NetKet 原生实现训练



100%|██████████| 400/400 [00:20<00:00, 19.77it/s, Energy=-1.015e+00-1.074e-09j ± 2.236e-10 [σ²=5.038e-17, R̂=1.4086]]



NetKet 训练完成！最终能量: -1.01546825 Ha

